# CellOracle reference GRN for Paul et al. 2015 data\n\nThis notebook produces a genuine CellOracle GRN for the Paul et al. 2015 hematopoiesis dataset, to serve as the comparison target in `compare_to_celloracle.ipynb`.\n\n**Why this notebook exists instead of just downloading CellOracle's tutorial output:** CellOracle's tutorial page (https://morris-lab.github.io/CellOracle.documentation/notebooks/04_Network_analysis/Network_analysis_with_Paul_etal_2015_data.html) does not ship a downloadable precomputed GRN for Paul15 — the GRN itself is computed in the (unlinked/prior) `03_GRN_calculation` notebook and only consumed in `04_Network_analysis`. So \"retrieving\" CellOracle's real result means re-running their pipeline, not fetching a file.\n\n**Important correction vs. the previous comparison (`analysis7.ipynb`):** the earlier `cell_oracle_run` results used to compare against were *not* run on Paul15 data at all — they used a different RNA dataset (Velten et al. hematopoiesis), a hand-built base GRN from custom ATAC motif files, and generic Leiden clusters. That comparison was apples-to-oranges regardless of how the Jaccard/Spearman metrics were computed. This notebook fixes that by using the real thing.\n\n**What "the real thing" turned out to already exist:** `/home/elicanter/datasets/paul2015data/Paul_15_data.celloracle.oracle` is a genuine checkpoint from CellOracle's own official tutorial pipeline — built with `co.data.load_Paul2015_data()`-equivalent preprocessing, the official `load_mouse_scATAC_atlas_base_GRN()` base GRN, and the official `TF_data_in_Paul15.csv` TF-target dictionary. It has the same 2671 cells, in the same order, as the `Paul_etal_15_2.h5ad` file your own model (COMET) was trained on (`config.py`'s `rna` path) — and its TF universe (90 regulators present in both the TF dict and the scRNA-seq data) matches your model's `A_tfs` count exactly. The checkpoint stops right before GRN inference (PCA / imputation / `get_links` not yet run), so this notebook finishes the official pipeline from that point, using library-default parameters throughout (nothing hand-tuned).\n\nKernel: `celloracle_env3` (the only local env where `celloracle` imports cleanly — `celloracle_env` is missing a system `libtiff` needed by Pillow).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import celloracle as co

print("celloracle version:", co.__version__)

ORACLE_CHECKPOINT = "/home/elicanter/datasets/paul2015data/Paul_15_data.celloracle.oracle"
OUT_DIR = "data/paul15/celloracle_reference"
os.makedirs(OUT_DIR, exist_ok=True)

oracle = co.load_hdf5(ORACLE_CHECKPOINT)
oracle

celloracle version: 0.20.0


Oracle object

Meta data
    celloracle version used for instantiation: 0.20.0
    n_cells: 2671
    n_genes: 1999
    cluster_name: louvain_annot
    dimensional_reduction_name: X_draw_graph_fa
    n_target_genes_in_TFdict: 21259 genes
    n_regulatory_in_TFdict: 1093 genes
    n_regulatory_in_both_TFdict_and_scRNA-seq: 90 genes
    n_target_genes_both_TFdict_and_scRNA-seq: 1850 genes
    k_for_knn_imputation: NA
Status
    Gene expression matrix: Ready
    BaseGRN: Ready
    PCA calculation: Not finished
    Knn imputation: Not finished
    GRN calculation for simulation: Not finished

## Choice of GRN unit: `paul15_clusters`

CellOracle fits one GRN per cluster (a per-cluster Bayesian ridge regression of each target gene on its candidate TFs, restricted to TF-gene edges allowed by the base GRN). The checkpoint carries several candidate cluster columns (`louvain_annot` — 24 clusters produced by re-clustering during preprocessing — and `paul15_clusters` — the original cell-type calls from the Paul et al. 2015 paper itself). We use `paul15_clusters` since it's the dataset's own ground-truth annotation rather than a clustering artifact of this particular preprocessing run.

`paul15_clusters` carries two categories with essentially no cells left after the standard QC filtering done upstream (`11DC`: 1 cell, `19Lymph`: 1 cell — remnants of cells that were mostly dropped in preprocessing because DC/Lymphoid populations aren't reliably captured in this data, but a couple of stray cells kept this original label). A per-cluster ridge regression cannot be fit on 1 cell, so these are dropped before GRN inference — this mirrors what CellOracle's own preprocessing tutorial does explicitly (it drops Lymphoid/DC cell types for the same reason, using the `louvain_annot`-derived `cell_type` column).

In [2]:
MIN_CLUSTER_SIZE = 10  # can't fit a per-cluster regression on a handful of cells

vc = oracle.adata.obs["paul15_clusters"].value_counts()
print(vc)

keep_clusters = vc[vc >= MIN_CLUSTER_SIZE].index.tolist()
dropped = vc[vc < MIN_CLUSTER_SIZE]
print(f"\nDropping {len(dropped)} clusters with < {MIN_CLUSTER_SIZE} cells:")
print(dropped)

# save full cell metadata (all cells, before dropping) so the comparison notebook
# can align cluster labels to COMET's cell order regardless of which clusters we keep here
cell_meta = oracle.adata.obs[["paul15_clusters", "louvain_annot", "cell_type"]].copy()
cell_meta.index.name = "obs_name"
cell_meta.to_csv(f"{OUT_DIR}/cell_metadata.csv")

mask = oracle.adata.obs["paul15_clusters"].isin(keep_clusters)
oracle.adata = oracle.adata[mask].copy()
oracle.adata.obs["paul15_clusters"] = oracle.adata.obs["paul15_clusters"].cat.remove_unused_categories()

print(f"\nKept {oracle.adata.shape[0]} cells across {len(keep_clusters)} clusters for GRN inference")

14Mo       373
2Ery       329
13Baso     300
3Ery       246
15Mo       186
5Ery       180
6Ery       173
7MEP       167
16Neu      164
10GMP      153
4Ery       124
12Baso      69
8Mk         68
9GMP        63
1Ery        43
17Neu       22
18Eos        9
11DC         1
19Lymph      1
Name: paul15_clusters, dtype: int64

Dropping 3 clusters with < 10 cells:
18Eos      9
11DC       1
19Lymph    1
Name: paul15_clusters, dtype: int64

Kept 2660 cells across 16 clusters for GRN inference


## PCA + KNN imputation

Standard CellOracle preprocessing before GRN fitting: PCA on the (already log-normalized) expression matrix, then a balanced KNN smoothing of expression used only as the regression input (not persisted as "the" expression matrix). Parameters follow the library's own tutorial defaults: number of PCs picked via a variance-explained elbow (capped at 50), and `k = 2.5%` of the cell count.

In [3]:
oracle.perform_PCA()

# elbow: first point where the second derivative of cumulative explained variance
# flattens out below 0.002, capped at 50 PCs (matches CellOracle's own tutorial heuristic)
diffs = np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_)) > 0.002)
n_comps = np.where(diffs)[0][0]
n_comps = min(n_comps, 50)
print("n_comps:", n_comps)

plt.plot(np.cumsum(oracle.pca.explained_variance_ratio_)[:100])
plt.axvline(n_comps, c="k", ls="--")
plt.xlabel("PC"); plt.ylabel("cumulative explained variance")
plt.show()

n_cell = oracle.adata.shape[0]
k = int(0.025 * n_cell)
print("n_cell:", n_cell, "k:", k)

oracle.knn_imputation(
    n_pca_dims=n_comps, k=k, balanced=True,
    b_sight=k * 8, b_maxl=k * 4, n_jobs=4,
)
print("KNN imputation done")

n_comps: 45
n_cell: 2660 k: 66


KNN imputation done


## GRN inference

`get_links` fits, independently per cluster, a Bayesian ridge regression of each target gene's (imputed) expression on the expression of its candidate TFs — where "candidate TFs" for a given gene are exactly the TFs whose motif was found in an accessible peak linked to that gene in the base GRN (the ATAC-derived prior). `alpha=10` is CellOracle's tutorial-default ridge regularization strength.

This is the step that actually defines what a "CellOracle GRN" *is*: for each cluster, a full table of `(source TF, target gene, coef_mean, coef_abs, p, -logp)` rows — one row per TF-target pair permitted by the base GRN, not a dense matrix. Edges with a zero coefficient are regression results too (the ridge fit shrank them to exactly zero or the base GRN never proposed that edge as a candidate), not missing data.

In [4]:
links = oracle.get_links(
    cluster_name_for_GRN_unit="paul15_clusters",
    alpha=10,
    verbose_level=1,
    n_jobs=4,
)

oracle.to_hdf5(f"{OUT_DIR}/Paul_15_data.grn.celloracle.oracle")
links.to_hdf5(f"{OUT_DIR}/Paul_15_data.celloracle.links")

print("clusters with a fitted GRN:", list(links.links_dict.keys()))
print("edges in first cluster's raw table:", len(links.links_dict[list(links.links_dict.keys())[0]]))

  0%|          | 0/16 [00:00<?, ?it/s]

clusters with a fitted GRN: ['10GMP', '12Baso', '13Baso', '14Mo', '15Mo', '16Neu', '17Neu', '1Ery', '2Ery', '3Ery', '4Ery', '5Ery', '6Ery', '7MEP', '8Mk', '9GMP']
edges in first cluster's raw table: 74937


## Filtering and network scores

`filter_links` is what actually makes each cluster's GRN sparse: it keeps edges with regression p-value below `p`, then keeps only the top `threshold_number` edges per cluster ranked by `coef_abs`. We use CellOracle's tutorial values (`p=0.001`, `weight="coef_abs"`, `threshold_number=2000`) rather than the library's more permissive built-in default (`10000`), since `2000` is what the published tutorial actually filters to and is a fairer stand-in for "CellOracle's real output."

`get_network_score` then computes, per gene per cluster, standard network-centrality statistics (in/out/total degree, degree centrality, betweenness centrality, eigenvector centrality) over each cluster's filtered directed graph. This is the metric CellOracle's own tutorial uses to call out "hub"/master regulators — it's a better ground truth for a hub-TF comparison than any hand-summed edge-weight score, since it's literally what the tool computes and reports for that purpose.

In [5]:
links.filter_links(p=0.001, weight="coef_abs", threshold_number=2000)
links.get_network_score()

for cl, df in links.filtered_links.items():
    print(f"{cl:10s}  filtered edges: {len(df):5d}   unique TFs: {df['source'].nunique():3d}   unique targets: {df['target'].nunique():4d}")

links.merged_score.to_csv(f"{OUT_DIR}/network_scores.csv")
links.merged_score.head()

10GMP       filtered edges:  2000   unique TFs:  85   unique targets:  491
12Baso      filtered edges:  2000   unique TFs:  83   unique targets:  447
13Baso      filtered edges:  2000   unique TFs:  82   unique targets:  522
14Mo        filtered edges:  2000   unique TFs:  85   unique targets:  522
15Mo        filtered edges:  2000   unique TFs:  85   unique targets:  524
16Neu       filtered edges:  2000   unique TFs:  82   unique targets:  493
17Neu       filtered edges:  2000   unique TFs:  81   unique targets:  484
1Ery        filtered edges:  2000   unique TFs:  59   unique targets:  577
2Ery        filtered edges:  2000   unique TFs:  76   unique targets:  542
3Ery        filtered edges:  2000   unique TFs:  79   unique targets:  538
4Ery        filtered edges:  2000   unique TFs:  76   unique targets:  558
5Ery        filtered edges:  2000   unique TFs:  80   unique targets:  530
6Ery        filtered edges:  2000   unique TFs:  80   unique targets:  499
7MEP        filtered edge

,degree_all,degree_centrality_all,degree_in,degree_centrality_in,degree_out,degree_centrality_out,betweenness_centrality,eigenvector_centrality,cluster
Gata1,69,0.125912,4,0.007299,65,0.118613,3235.0,0.779287,10GMP
Ctsg,5,0.009124,5,0.009124,0,0.000000,0.0,0.671167,10GMP
Gata2,152,0.277372,9,0.016423,143,0.260949,5653.0,1.000000,10GMP
Spi1,83,0.151460,4,0.007299,79,0.144161,1382.0,0.504765,10GMP
Hspa5,1,0.001825,1,0.001825,0,0.000000,0.0,0.119008,10GMP


## Export artifacts for the comparison notebook

Saves, per cluster, the filtered edge table (`source`, `target`, `coef_mean`, `coef_abs`, `p`, `-logp`) as CSV, plus the base-GRN gene/TF universe. Kept as long tables (not pivoted to a dense matrix here) so the comparison notebook can decide how to handle structural zeros explicitly rather than inheriting a pivot's `fillna(0)`.

In [6]:
edges_dir = f"{OUT_DIR}/edges_by_cluster"
os.makedirs(edges_dir, exist_ok=True)

for cl, df in links.filtered_links.items():
    safe_name = str(cl).replace("/", "_")
    df.to_csv(f"{edges_dir}/{safe_name}.csv", index=False)

# base-GRN gene/TF universe (the full candidate space GRN edges were drawn from)
base_grn_genes = np.array(sorted(oracle.adata.var_names[oracle.adata.var["isin_TFdict_targets"]]))
base_grn_tfs = np.array(sorted(oracle.adata.var_names[oracle.adata.var["isin_TFdict_regulators"]]))
np.save(f"{OUT_DIR}/base_grn_genes.npy", base_grn_genes)
np.save(f"{OUT_DIR}/base_grn_tfs.npy", base_grn_tfs)

print(f"base GRN universe: {len(base_grn_genes)} target genes, {len(base_grn_tfs)} TFs")
print(f"\nSaved to {OUT_DIR}/:")
for f in sorted(os.listdir(OUT_DIR)):
    print(" ", f)

base GRN universe: 1850 target genes, 90 TFs

Saved to data/paul15/celloracle_reference/:
  Paul_15_data.celloracle.links
  Paul_15_data.grn.celloracle.oracle
  base_grn_genes.npy
  base_grn_tfs.npy
  cell_metadata.csv
  edges_by_cluster
  network_scores.csv


## Sanity check: known hematopoietic regulators should show up as hubs

`Gata1`/`Klf1` (erythroid), `Spi1`/`Cebpa` (myeloid) are well-established master regulators in this system — CellOracle's own tutorial highlights them. If they don't show elevated eigenvector centrality here, something upstream is off (wrong cluster column, bad imputation, etc.) before we even get to comparing against COMET.

In [7]:
known_regulators = ["Gata1", "Klf1", "Spi1", "Cebpa", "Gata2", "Fli1"]

hub_summary = (
    links.merged_score
    .reset_index().rename(columns={"index": "gene"})
    .groupby("gene")["eigenvector_centrality"].median()
    .sort_values(ascending=False)
)

print("Overall top-15 hub TFs (median eigenvector centrality across clusters):")
print(hub_summary.head(15))

print("\nRank of known hematopoietic regulators:")
ranks = hub_summary.rank(ascending=False)
for g in known_regulators:
    if g in ranks.index:
        print(f"  {g:8s} rank {int(ranks[g]):4d} / {len(ranks)}   score={hub_summary[g]:.4f}")
    else:
        print(f"  {g:8s} not in merged_score (not a regulator in any cluster's filtered GRN)")

Overall top-15 hub TFs (median eigenvector centrality across clusters):
gene
Gata2      0.741984
Myc        0.396626
Nfe2       0.381339
Irf8       0.369560
Gata1      0.329654
Apoe       0.291867
E2f4       0.270175
Nr3c1      0.261543
Myb        0.256986
Zbtb7a     0.256316
Egr1       0.251175
Elf1       0.234567
Fli1       0.224199
Fam132a    0.217563
Gstm1      0.214079
Name: eigenvector_centrality, dtype: float64

Rank of known hematopoietic regulators:
  Gata1    rank    5 / 1125   score=0.3297
  Klf1     rank   22 / 1125   score=0.2013
  Spi1     rank   41 / 1125   score=0.1361
  Cebpa    rank  126 / 1125   score=0.0754
  Gata2    rank    1 / 1125   score=0.7420
  Fli1     rank   13 / 1125   score=0.2242
